# Reference · Day 6 studio — how much does shuffling flatter you?

**Not a marking key.** You chose your own third model, and you may have cut the bike
data somewhere else, so your numbers are not meant to match these. What is worth
comparing is the *shape*: whether shuffled CV beat the forward split for your model,
whether dropping `yr` did what you predicted, and whether you said so when Ames had
nothing to show at month resolution.

The studio ran the three-step check from the meeting — cross-validate with random
folds, fit on the past and score the future, compare — three more times. Every
section below is that check, plus one thing the meeting did not do: **asking whether
the gap is bigger than the noise.**

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

bike = load("bike_hour.csv")
bike["dteday"] = pd.to_datetime(bike["dteday"])
FEAT = [c for c in bike.columns
        if c not in ("cnt", "casual", "registered", "dteday", "instant")]

ames = load("ames.csv")
NUM = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
CAT = ["Neighborhood", "Central_Air"]


def bike_pipe(model):
    return Pipeline([("fill", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", model)])


def ames_pipe(model=None):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                     fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]), CAT),
        ])),
        ("model", model or LinearRegression()),
    ])


print(f"  bike: {len(bike):,} hours, {bike['dteday'].min():%b %Y} to {bike['dteday'].max():%b %Y}")
print(f"  ames: {len(ames):,} sales, {ames['Yr_Sold'].min()} to {ames['Yr_Sold'].max()}")

## One check, used for everything

The meeting's three steps, written **once** as a function, so that every row below is
scored the same way. If each row had its own copy of the code, a difference between
two rows could be a difference in the code.

In [ ]:
MODELS = {
    "OLS": lambda: LinearRegression(),
    "RandomForest": lambda: RandomForestRegressor(n_estimators=60, random_state=0,
                                                  n_jobs=-1),
    "GradientBoosting": lambda: GradientBoostingRegressor(random_state=0),
}


_shuffled = {}          # step 1 does not depend on the cut, so compute it once


def check(name, feat=FEAT, cut=None):
    """1. cross-validate with random folds  2. fit past, score future  3. compare."""
    cut = bike["dteday"].quantile(0.75) if cut is None else cut
    before, after = bike[bike["dteday"] <= cut], bike[bike["dteday"] > cut]
    key = (name, tuple(feat))
    if key not in _shuffled:
        _shuffled[key] = cross_val_score(bike_pipe(MODELS[name]()), bike[feat], bike["cnt"],
                                         cv=KFold(5, shuffle=True, random_state=0),
                                         scoring="r2").mean()
    shuffled = _shuffled[key]
    forward = r2_score(after["cnt"], bike_pipe(MODELS[name]())
                       .fit(before[feat], before["cnt"]).predict(after[feat]))
    return {"model": name, "random CV": shuffled, "forward": forward,
            "optimism": shuffled - forward}

## Question 1 — a third model. Does the optimism depend on the model?

The meeting cut the bike data at its 75th percentile date and got **+0.083** for OLS
and **+0.050** for the forest. Here is a third model on the same cut.

In [ ]:
q1 = pd.DataFrame([check(name) for name in MODELS])
print(q1.round(3).to_string(index=False))

On this cut, the forest is the least flattered and gradient boosting is flattered about
as much as OLS. It would be easy to write *"forests are more robust to drift"* and stop.

⚠ **That is one split.** Sep 15 and Sep 24 both said what one split is worth. The date
we cut at is as much a choice as `random_state` was, so move it:

In [ ]:
sweep = []
for q in (0.6, 0.7, 0.75, 0.8, 0.9):
    cut = bike["dteday"].quantile(q)
    for name in MODELS:
        sweep.append({"cut at": f"{q:.0%} ({cut:%b %d %Y})", **check(name, cut=cut)})
sweep = pd.DataFrame(sweep)
table = sweep.pivot(index="cut at", columns="model", values="optimism")
print("  optimism = random CV minus forward, by where the data is cut\n")
print(table.round(3).to_string())
print(f"\n  shuffled CV beat the forward split in {int((sweep['optimism'] > 0).sum())} "
      f"of {len(sweep)} cases")
print("  least flattered model at each cut: "
      + ", ".join(f"{c[:3]}: {m}" for c, m in table.idxmin(axis=1).items()))

**Yes, the size depends on the model — and on where you cut, and the two interact.**
The forest is the least flattered at four cuts out of five. At the fifth, the 60% cut,
it is flattered *more* than OLS, 0.177 against 0.114, and gradient boosting is flattered
most of all at 0.214.

So there are two claims here, and they are not the same kind:

- **Random CV was optimistic in fifteen cases out of fifteen** — every model, every
  cut. That is a finding.
- **The forest is least flattered** in four out of five, with one reversal. That is a
  lead worth checking on new data, not a property of forests.

## Is the gap bigger than the noise?

The meeting compared two numbers and called one gap big and the other nothing. Neither
was compared with anything. Here is the yardstick: hold out a **random** set of rows the
same size as the forward test set, train on the rest, and repeat. If the forward score
sits inside that spread, the gap is noise.

In [ ]:
def random_holdouts(frame, n_test, fit_score, reps, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(reps):
        test = np.zeros(len(frame), bool)
        test[rng.choice(len(frame), n_test, replace=False)] = True
        out.append(fit_score(frame[~test], frame[test]))
    return np.array(out)


def bike_score(name):
    return lambda tr, te: r2_score(te["cnt"], bike_pipe(MODELS[name]())
                                   .fit(tr[FEAT], tr["cnt"]).predict(te[FEAT]))


def ames_score(tr, te):
    return r2_score(te["SalePrice"], ames_pipe()
                    .fit(tr[NUM + CAT], tr["SalePrice"]).predict(te[NUM + CAT]))


cut = bike["dteday"].quantile(0.75)
n_after = int((bike["dteday"] > cut).sum())
flat = bike.reset_index(drop=True)
print(f"  forward vs random holdouts of the SAME size ({n_after:,} rows)\n")
for name, reps in [("OLS", 30), ("RandomForest", 10)]:
    fwd = q1.set_index("model").loc[name, "forward"]
    r = random_holdouts(flat, n_after, bike_score(name), reps)
    print(f"  bike {name:<13} forward {fwd:.3f}   random: mean {r.mean():.3f}, "
          f"SD {r.std(ddof=1):.3f}, range {r.min():.3f} to {r.max():.3f}   "
          f"-> {(r.mean() - fwd) / r.std(ddof=1):.0f} SD below")

y2010 = ames["Yr_Sold"] == 2010
fwd_ames = ames_score(ames[~y2010], ames[y2010])
r = random_holdouts(ames.reset_index(drop=True), int(y2010.sum()), ames_score, 30)
print(f"\n  ames OLS          forward {fwd_ames:.3f}   random: mean {r.mean():.3f}, "
      f"SD {r.std(ddof=1):.3f}, range {r.min():.3f} to {r.max():.3f}   "
      f"-> {(fwd_ames - r.mean()) / r.std(ddof=1):+.1f} SD")

**The bike gaps are real by a wide margin**: 8 standard deviations for OLS and 17 for the
forest. Random 4,328-row holdouts barely move, so a gap of 0.05 is enormous there.

**The Ames number is noise — and the check could not have seen much.** A random
341-house holdout swings by roughly ±0.07 on its own. So *"I checked and it was fine"*
means **no shift bigger than about 0.14**. That is still a finding. It is a smaller one
than it sounded like on Thursday, and the honest report says how small.

## Question 2 — drop `yr`. Better or worse?

`yr` is 0 for 2011 and 1 for 2012. The obvious argument: it encodes "2012 is busier",
a fact about the past that cannot be carried into the future, so removing it should
make the forward number *better*.

Commit to that guess — or its opposite — before you run the next cell.

In [ ]:
NO_YR = [c for c in FEAT if c != "yr"]
q2 = []
for name in ("OLS", "RandomForest"):
    for label, feat in [("with yr", FEAT), ("without yr", NO_YR)]:
        q2.append({"features": label, **check(name, feat)})
print(f"  cut at the 75th percentile: train through {cut:%b %d %Y}\n")
print(pd.DataFrame(q2).round(3).to_string(index=False))

**Worse, for both models.** OLS falls from 0.305 to 0.185; the forest from 0.894 to 0.663.

The obvious argument had the mechanism right and the situation wrong. This cut trains
through early **July 2012**, so the training rows already contain half a year of 2012.
`yr` is not an unknowable future here — it is a column the model has genuinely seen
both values of, and it is how the model knows the second half of the data is busier.

The argument *is* right where it applies: at a cut where the future year has never
been seen. So change one thing — the cut — and ask again.

In [ ]:
year_cut = pd.Timestamp("2011-12-31")
q2b = []
for name in ("OLS", "RandomForest"):
    for label, feat in [("with yr", FEAT), ("without yr", NO_YR)]:
        q2b.append({"features": label, **check(name, feat, cut=year_cut)})
print("  train 2011, predict 2012 -- yr is 0 in EVERY training row\n")
print(pd.DataFrame(q2b).round(3).to_string(index=False))
ols = pd.DataFrame(q2b).query("model == 'OLS'").set_index("features")
print(f"\n  yr is worth {ols.loc['with yr', 'random CV'] - ols.loc['without yr', 'random CV']:.3f}"
      " to the shuffled OLS score, and "
      f"{abs(ols.loc['with yr', 'forward'] - ols.loc['without yr', 'forward']):.3f}"
      " to the forward one")

**Identical forward numbers, for both models.** When every training row has `yr = 0`
there is nothing to learn from the column, so keeping it or dropping it cannot change a
single prediction. Its entire value — 0.049 of shuffled R-squared for OLS — is an
artifact of shuffling 2012 rows into training. That is the meeting's thesis inside one
column.

It also shows that **which forward split you choose is itself a decision.** Optimism is
+0.083 at the 75th percentile and +0.236 at the year boundary, because the year split
forces the model to predict a year it has never seen. Neither is *the* optimism of this
model. Each answers a different "how far ahead?"

## Question 3 — Ames by month

The studio asked: 2010 only, first nine months to predict the rest. Look before you
fit.

In [ ]:
print(pd.crosstab(ames["Yr_Sold"], ames["Mo_Sold"]).to_string())

**The data stop in July 2010**, with eight sales that month. There is no "rest" after
nine months. The right first answer to Question 3 is *"this cannot be asked of this
data"* — and producing a number anyway is the one wrong answer.

What *can* be asked is a nearby question, changing one thing at a time. Within 2010,
train on January–April and predict May–July. But that changes two things at once
compared with the meeting: time **and** training size, because 172 houses is far fewer
than 2,589. So the yardstick here is a random split **of the same 2010 houses, with the
same sizes** — time is the only thing that differs.

In [ ]:
a10 = ames[y2010].reset_index(drop=True)
early, late = a10[a10["Mo_Sold"] <= 4], a10[a10["Mo_Sold"] >= 5]
fwd10 = ames_score(early, late)
r10 = random_holdouts(a10, len(late), ames_score, 30)
lo, hi = np.percentile(r10, [5, 95])
print(f"  2010 Jan-Apr ({len(early)}) -> May-Jul ({len(late)}):   forward {fwd10:.3f}")
print(f"  random splits of the same 2010 houses, same sizes:   median {np.median(r10):.3f}, "
      f"90% of them between {lo:.3f} and {hi:.3f}")
below = int((r10 < fwd10).sum())
chance = (len(r10) - below + 1) / (len(r10) + 1)       # rank this high, if time did nothing
print(f"  the forward split scored higher than {below} of {len(r10)} random splits")
print(f"  if time made no difference, a rank this high happens about 1 time in {round(1 / chance)}")

**Above the band, not below it.** The forward split scored higher than 29 of the 30
random splits. If time had hurt prediction, the forward number would sit at the
*bottom* of that spread; it sits at the top.

Is that a finding? One comparison against thirty reference splits lands this high about
one time in sixteen by chance alone. It is certainly not evidence of drift that hurts
you, and it is too thin to be evidence that late spring is easier to price.

Now keep every earlier year in training and step through 2010 one month at a time — the
most data an assessor could have had. Each month's test set is 8 to 82 houses:

In [ ]:
ames["month"] = ames["Yr_Sold"] * 12 + ames["Mo_Sold"]
rolling = []
for m in range(1, 8):
    now = 2010 * 12 + m
    past, this = ames[ames["month"] < now], ames[ames["month"] == now]
    pool = ames[ames["month"] <= now].reset_index(drop=True)
    r = random_holdouts(pool, len(this), ames_score, 30, seed=m)
    lo, hi = np.percentile(r, [5, 95])
    rolling.append({"month": f"2010-{m:02d}", "houses": len(this),
                    "forward": ames_score(past, this),
                    "random 5%": lo, "random 95%": hi})
rolling = pd.DataFrame(rolling)
rolling["inside band"] = rolling["forward"].between(rolling["random 5%"],
                                                    rolling["random 95%"])
print(rolling.round(3).to_string(index=False))
print(f"\n  {int(rolling['inside band'].sum())} of {len(rolling)} months inside their own 90% band;"
      f" chance alone puts about {0.1 * len(rolling):.1f} outside")
width = rolling["random 95%"] - rolling["random 5%"]
print(f"  band widths: {width.min():.2f} to {width.max():.2f} of R-squared")

**Six of seven months sit inside their own noise band**, and January misses by 0.008 —
with seven months checked against a 90% band, about one miss is what chance delivers.
**And the bands are huge**: 0.15 to 0.46 of R-squared wide. A month-level shift would
have to be enormous to show up here.

⚠ **Resolution costs power.** Going from years to months made the question finer and the
answer weaker, because each test set got smaller. "Nothing at month resolution" is an
acceptable finding only if you say how little the check could see.

## What you would say, and what you would refuse to say

| | |
|---|---|
| **"Random CV overstated every model on the bike data"** | ✅ Fifteen cases out of fifteen, and 8 to 17 standard deviations beyond noise on the one cut we tested against it. |
| **"Forests are more robust to drift than OLS"** | ⚠ Least flattered at four cuts of five, reversed at the fifth. A lead, not a property — and one cut could never have told you. |
| **"Dropping `yr` fixes the leak"** | ❌ It made the forward number worse at the 75th percentile, and changed nothing at the year boundary. |
| **"Shuffling costs nothing on Ames"** | ⚠ Defensible only with the size attached: no shift larger than about 0.14. |
| **"Ames prices drift month to month"** | ❌ The month-level noise bands are 0.15 to 0.46 wide. The check could not have seen it. |
| **"Late-spring 2010 houses are easier to price"** | ❌ One split beat 29 of 30 random splits — about a one-in-sixteen event by chance alone. |

## Check your own notebook against this

Not "did I get these numbers". These:

| | |
|---|---|
| **1** | Before you dropped `yr`, did you **write down what you expected** — and keep it in the notebook when the measurement disagreed? |
| **2** | Before ranking models by optimism, did you **move the cut** at least once, or did you rank them on one split? |
| **3** | When you moved Ames to months, did you notice the question **could not be asked as written** — and say so rather than fitting something anyway? |
| **4** | When you compared two situations, did you **change one thing at a time**, or did the training size change along with the time? |
| **5** | When you called a gap "big" or "nothing", did you compare it with **how much a random holdout of the same size moves on its own**? |

If 5 is a no, that is the one to take into Lab 4 and the capstone evaluation plan.
**A gap is only big relative to the noise, and a check is only reassuring relative to
what it could have seen.**